# 11 Scaling Enterprise RAG & Security

**Real-World Scenario**: Multi-Tenant Enterprise HR Portal with Strict Role-Based Security (ACLs).

This notebook demonstrates **Role-Based ACL Metadata Payload Filtering** in LangChain, saving vector indexes into a hidden `.vectordb/` folder.

## Step 1: Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from repository root .env
load_dotenv()

# Create hidden vector database directory for local index persistence
os.makedirs(".vectordb", exist_ok=True)

print("=== Repository Environment Setup ===")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))
print("Hidden DB Directory Path:", os.path.abspath(".vectordb"))


## Step 2: Multi-Tenant Role-Based ACL Payload Filtering Code

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

docs = [
    "Public Employee Handbook: All staff receive 20 days paid vacation.",
    "Manager Salary Budget: Senior manager salary band is $180k-$220k.",
    "Executive Acquisition Plan: Project Titan acquisition budget $50M."
]
metadatas = [
    {"role": "Public"},
    {"role": "Manager"},
    {"role": "Executive"}
]

if os.getenv("OPENAI_API_KEY"):
    vectorstore = FAISS.from_texts(docs, OpenAIEmbeddings(), metadatas=metadatas)
    save_path = os.path.join(".vectordb", "11_acl_index")
    vectorstore.save_local(save_path)
    
    # Manager role querying vector store
    results = vectorstore.similarity_search("Handbook Salary", k=2, filter={"role": "Manager"})
    print(f"=== Multi-Tenant ACL Payload Filtered Results (Role = Manager) ===")
    for r in results:
        print(f"Role Tag: {r.metadata['role']} | Content: {r.page_content}")
    print(f"Saved ACL index to hidden folder: {save_path}")
